# Activity Hub Project Update Validation Against WM Week Calendar

This notebook validates which projects have been updated in the current Walmart Week by:
1. Loading the WM Week Calendar (Cal Dim Cur) from BigQuery
2. Comparing Project Update Dates against the calendar
3. Identifying projects with updates in the current WM Week vs previous weeks

**Key Question:** Are projects marked "Updated" only if their Project Update Date falls in the current WM Week?

## Section 1: Import Required Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from google.cloud import bigquery
import json

# Setup Google Cloud credentials
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = r'C:\Users\krush\AppData\Roaming\gcloud\application_default_credentials.json'

# Initialize BigQuery client
bq_client = bigquery.Client(project='wmt-assetprotection-prod')

print("✓ Libraries imported successfully")
print(f"✓ BigQuery client initialized for project: {bq_client.project}")

## Section 2: Connect to Knowledge Base and Load Cal Dim Cur Data

Load the Walmart Week Calendar (Cal Dim Cur) from the enterprise data warehouse.

In [ ]:
# Query to load Walmart Week Calendar dates (Cal_Dim_Cur)
calendar_query = """
SELECT 
    MIN(CAL_DT) as week_start,
    MAX(CAL_DT) as week_end,
    WM_WK_NBR,
    WM_YR_NBR,
    COUNT(*) as num_days
FROM `polaris-analytics-prod.us_walmart.Cal_Dim_Cur`
WHERE WM_YR_NBR = 2026
  AND WM_WK_NBR IN (11, 12)
GROUP BY WM_WK_NBR, WM_YR_NBR
ORDER BY WM_WK_NBR
"""

try:
    cal_results = bq_client.query(calendar_query).to_dataframe()
    print("✓ Calendar dimension loaded from Cal_Dim_Cur")
    print("\nWalmart Week 11 & 12 Date Ranges:")
    print(cal_results.to_string(index=False))
    
    # Extract the dates for our analysis
    wk11_start = cal_results[cal_results['WM_WK_NBR'] == 11]['week_start'].values[0]
    wk11_end = cal_results[cal_results['WM_WK_NBR'] == 11]['week_end'].values[0]
    wk12_start = cal_results[cal_results['WM_WK_NBR'] == 12]['week_start'].values[0]
    wk12_end = cal_results[cal_results['WM_WK_NBR'] == 12]['week_end'].values[0]
    
    print(f"\n📅 WM WK 11: {wk11_start} to {wk11_end}")
    print(f"📅 WM WK 12: {wk12_start} to {wk12_end}")
    
except Exception as e:
    print(f"⚠ Error loading Cal_Dim_Cur: {e}")
    cal_results = None

## Section 3: Parse and Understand WM Week Calendar

Understand the structure of the Walmart Week calendar and how dates map to WM weeks.

In [ ]:
if cal_dim is not None and len(cal_dim) > 0:
    # Using actual calendar dimension
    print("📊 WM Week Calendar Structure (from Cal Dim Cur):\n")
    
    # Group by WM_WEEK_NBR to see the date range for each week
    week_summary = cal_dim.groupby('WM_WEEK_NBR').agg({
        'CALENDAR_DATE': ['min', 'max', 'count']
    }).reset_index()
    week_summary.columns = ['WM_WEEK', 'Week_Start', 'Week_End', 'Num_Days']
    
    print(week_summary)
    print(f"\nWalmart Week duration: Varies by week (includes partial weeks)")
else:
    # Using calculated method based on Fiscal Year
    print("📊 WM Week Calculation Method:\n")
    print("  Walmart Fiscal Year starts: February 1")
    print("  WM Week calculation: (days_since_FY_start // 7) + 1")
    print("  Example:")
    print("    - Feb 1: Day 0 → WK1")
    print("    - Feb 8: Day 7 → WK2")
    print("    - Feb 15: Day 14 → WK3")
    print("    - ... and so on")
    print("\nThis ensures consistent week boundaries across the fiscal year")

## Section 4: Extract Current WM Week

Determine what the current Walmart Week is today.

In [ ]:
# Function to calculate WM Week from a date
def get_wm_week(date):
    """
    Calculate WM (Walmart) Week number based on fiscal year starting Feb 1.
    Formula: (days_since_fiscal_year_start // 7) + 1
    """
    if isinstance(date, pd.Timestamp):
        date = date.to_pydatetime()
    
    # Fiscal year starts Feb 1
    fy_start = datetime(date.year if date.month >= 2 else date.year - 1, 2, 1)
    days_since_fy = (date.replace(tzinfo=None) - fy_start).days
    wm_week = (days_since_fy // 7) + 1
    return wm_week

# Get today's date and current WM week
today = datetime.now()
current_wm_week = get_wm_week(today)
fy_start = datetime(today.year if today.month >= 2 else today.year - 1, 2, 1)

print(f"📅 TODAY'S DATE & WM WEEK:")
print(f"   Today: {today.strftime('%Y-%m-%d (%A)')}")
print(f"   Fiscal Year Start: {fy_start.strftime('%Y-%m-%d')}")
print(f"   Days Since FY Start: {(today - fy_start).days}")
print(f"   ➤ CURRENT WM WEEK: WK{current_wm_week}")
print(f"\n   This means projects updated between {(today - timedelta(days=(today - fy_start).days % 7)).strftime('%Y-%m-%d')} and {(today + timedelta(days=6 - ((today - fy_start).days % 7))).strftime('%Y-%m-%d')} are in WK{current_wm_week}")

## Section 5: Extract Project Update Date and WM Week

Retrieve all projects from Activity Hub and map their update dates to WM weeks.

In [ ]:
# Query Activity Hub Projects with update dates
projects_query = """
SELECT 
    project_id,
    title,
    project_update_date,
    project_update_by,
    health,
    status
FROM `wmt-assetprotection-prod.Store_Support_Dev.AH_Projects`
WHERE project_update_date IS NOT NULL
ORDER BY project_update_date DESC
"""

try:
    projects_df = bq_client.query(projects_query).to_dataframe()
    print(f"✓ Projects loaded: {len(projects_df)} projects with update dates\n")
    
    # Calculate WM week for each project update date
    projects_df['Update_WM_Week'] = projects_df['project_update_date'].apply(get_wm_week)
    
    # Display sample
    print("Sample of projects with their update dates and WM weeks:")
    sample_cols = ['project_id', 'title', 'project_update_date', 'Update_WM_Week', 'health']
    print(projects_df[sample_cols].head(10).to_string())
    
except Exception as e:
    print(f"✗ Error loading projects: {e}")
    projects_df = None

## Section 6: Compare Update WM Week to Current WM Week

Compare each project's update week against the current WM week.

In [ ]:
if projects_df is not None and len(projects_df) > 0:
    # Compare each project's update week with current week
    projects_df['Is_Current_Week'] = projects_df['Update_WM_Week'] == current_wm_week
    projects_df['Update_Status'] = projects_df['Is_Current_Week'].apply(
        lambda x: f'✓ UPDATED (WK{current_wm_week})' if x else '✗ NOT UPDATED'
    )
    
    print(f"🔍 COMPARISON RESULTS (Current WM Week: WK{current_wm_week}):\n")
    
    # Display the full comparison
    display_cols = ['project_id', 'title', 'project_update_date', 'Update_WM_Week', 'Update_Status', 'health']
    comparison_df = projects_df[display_cols].copy()
    comparison_df.columns = ['Project ID', 'Title', 'Update Date', 'Update WK', 'Status', 'Health']
    
    # Format for display
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', None)
    pd.set_option('display.width', None)
    
    print(comparison_df.to_string(index=False))
else:
    print("No projects to compare")

## Section 7: Identify Updated vs Not Updated Projects

Classify projects into "Updated" and "Not Updated" categories.

In [ ]:
if projects_df is not None and len(projects_df) > 0:
    # Separate updated and not updated projects
    updated_projects = projects_df[projects_df['Is_Current_Week']].copy()
    not_updated_projects = projects_df[~projects_df['Is_Current_Week']].copy()
    
    print(f"📊 PROJECT CLASSIFICATION:\n")
    print(f"{'='*80}")
    print(f"\n✓ PROJECTS UPDATED IN CURRENT WM WEEK (WK{current_wm_week}):\n")
    print(f"   Count: {len(updated_projects)} projects")
    
    if len(updated_projects) > 0:
        print(f"\n   Project Details:")
        for idx, row in updated_projects.iterrows():
            print(f"   • ID: {row['project_id']} | {row['title'][:50]}")
            print(f"     Updated: {row['project_update_date']} by {row['project_update_by']}")
            print(f"     Health: {row['health']}")
            print()
    
    print(f"\n✗ PROJECTS NOT UPDATED IN CURRENT WEEK (Updated Previously):\n")
    print(f"   Count: {len(not_updated_projects)} projects")
    
    if len(not_updated_projects) > 0:
        print(f"\n   Top 10 Most Recent (by update date):")
        for idx, row in not_updated_projects.head(10).iterrows():
            weeks_ago = current_wm_week - row['Update_WM_Week']
            print(f"   • ID: {row['project_id']} | {row['title'][:50]}")
            print(f"     Updated: {row['project_update_date']} (WK{row['Update_WM_Week']}) - {weeks_ago} weeks ago")
            print(f"     Health: {row['health']}")
            print()
    
    print(f"{'='*80}")
    print(f"\nSummary:")
    print(f"   Total Projects: {len(projects_df)}")
    print(f"   Updated This Week: {len(updated_projects)} ({100*len(updated_projects)/len(projects_df):.1f}%)")
    print(f"   Not Updated This Week: {len(not_updated_projects)} ({100*len(not_updated_projects)/len(projects_df):.1f}%)")
else:
    print("No projects data available")

## Section 8: Generate Summary Report

Create a comprehensive summary report of project update status.

In [ ]:
if projects_df is not None and len(projects_df) > 0:
    print("\n" + "="*80)
    print("📋 ACTIVITY HUB PROJECT UPDATE VALIDATION REPORT")
    print("="*80)
    print(f"\nReport Date: {today.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Current WM Week: WK{current_wm_week}")
    print(f"Fiscal Year: FY{today.year if today.month >= 2 else today.year - 1}")
    print(f"\n{'-'*80}")
    
    # Summary metrics
    updated_count = len(updated_projects)
    not_updated_count = len(not_updated_projects)
    total_count = len(projects_df)
    
    print(f"\n📊 KEY METRICS:")
    print(f"   Total Projects with Update Dates: {total_count}")
    print(f"   ✓ Updated This Week (WK{current_wm_week}): {updated_count} ({100*updated_count/total_count:.1f}%)")
    print(f"   ✗ Not Updated This Week: {not_updated_count} ({100*not_updated_count/total_count:.1f}%)")
    
    # Health Status Breakdown
    print(f"\n💡 HEALTH STATUS BREAKDOWN:")
    health_breakdown = projects_df.groupby(['health', 'Is_Current_Week']).size().unstack(fill_value=0)
    print(health_breakdown)
    
    # Update frequency distribution
    print(f"\n📈 UPDATE FREQUENCY (by WM Week):")
    week_dist = projects_df['Update_WM_Week'].value_counts().sort_index(ascending=False).head(10)
    for wk, count in week_dist.items():
        current_marker = " ← CURRENT" if wk == current_wm_week else ""
        print(f"   WK{wk}: {count} projects{current_marker}")
    
    # Export summary
    print(f"\n💾 DATA EXPORT:")
    export_df = projects_df[['project_id', 'title', 'project_update_date', 'Update_WM_Week', 
                             'Is_Current_Week', 'health', 'status', 'project_update_by']].copy()
    export_df.columns = ['ProjectID', 'Title', 'UpdateDate', 'UpdateWK', 'IsCurrentWeek', 'Health', 'Status', 'UpdatedBy']
    
    # Save to CSV
    export_path = r'c:\Users\krush\OneDrive - Walmart Inc\Documents\VSCode\Activity_Hub\project_update_validation.csv'
    export_df.to_csv(export_path, index=False)
    print(f"   ✓ Report exported to: {export_path}")
    
    print(f"\n{'='*80}\n")
else:
    print("⚠ Unable to generate report - no projects data available")

In [ ]:
# Compare SOURCE data vs SYNCED data
print("\n" + "="*100)
print("COMPARISON: Source vs Synced Data")
print("="*100)

# Query source table
source_query = """
SELECT COUNT(DISTINCT Intake_Card_Nbr) as count_wk12,
       MIN(Project_Update_Date) as earliest,
       MAX(Project_Update_Date) as latest
FROM `wmt-assetprotection-prod.Store_Support_Dev.Output - Intake Accel Council Data`
WHERE Project_Update_Date > '2026-04-18'
  AND Project_Update_Date IS NOT NULL
"""

# Query synced table
synced_query = """
SELECT COUNT(DISTINCT project_id) as count_wk12,
       MIN(project_update_date) as earliest,
       MAX(project_update_date) as latest
FROM `wmt-assetprotection-prod.Store_Support_Dev.AH_Projects`
WHERE project_update_date > '2026-04-18'
  AND project_update_date IS NOT NULL
"""

try:
    source_results = bq_client.query(source_query).to_dataframe()
    synced_results = bq_client.query(synced_query).to_dataframe()
    
    print("\nSOURCE TABLE (Intake Accel Council Data):")
    print(f"  Projects updated > 2026-04-18: {source_results['count_wk12'].values[0]}")
    print(f"  Date range: {source_results['earliest'].values[0]} to {source_results['latest'].values[0]}")
    
    print("\nSYNCED TABLE (AH_Projects):")
    print(f"  Projects updated > 2026-04-18: {synced_results['count_wk12'].values[0]}")
    print(f"  Date range: {synced_results['earliest'].values[0]} to {synced_results['latest'].values[0]}")
    
    discrepancy = synced_results['count_wk12'].values[0] - source_results['count_wk12'].values[0]
    print(f"\n⚠️  DISCREPANCY: {discrepancy} more projects in SYNCED table")
    
except Exception as e:
    print(f"Error comparing data: {e}")